# Ajuste fino LoRA de Stable Diffusion con `diffusers`

El cuaderno adapta Stable Diffusion v1.5 al conjunto público Pokémon BLIP captions mediante LoRA. El modelo base permanece congelado; se entrenan matrices de rango bajo en atención. Se comparan generaciones con una misma semilla y prompt antes/después.

**Requisitos:** activar GPU en Colab; aceptar previamente la licencia del checkpoint Stable Diffusion v1.5 en Hugging Face y guardar un token personal como secreto `HF_TOKEN` en Colab. No escriba el token en el cuaderno ni en GitHub. El entrenamiento es exploratorio (300 pasos), no una garantía de calidad final. La licencia de modelo y conjunto debe revisarse antes de reutilizar imágenes fuera de la clase.

In [ ]:
!nvidia-smi
!pip -q install -U diffusers transformers datasets accelerate peft bitsandbytes huggingface_hub
!git clone --depth 1 https://github.com/huggingface/diffusers.git /content/diffusers
!pip -q install -e /content/diffusers


In [ ]:
import os, torch, matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import login
from google.colab import userdata

if not torch.cuda.is_available():
    raise RuntimeError('Activa un entorno GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución.')
try:
    token = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Agrega HF_TOKEN en Secretos de Colab y habilita su acceso a este cuaderno.') from exc
if not token:
    raise RuntimeError('No se encontró HF_TOKEN en los Secretos de Colab.')
login(token=token, add_to_git_credential=False)
print('GPU:', torch.cuda.get_device_name(0))


## 1. Inspección del conjunto

Pokémon BLIP captions contiene pares imagen-texto, columnas `image` y `text`. Se inspeccionan ejemplos antes del ajuste para confirmar que captions e imágenes se corresponden. La partición de validación queda separada para mirar progreso, no para cambiar el checkpoint manualmente tras cada imagen.

In [ ]:
dataset = load_dataset('lambdalabs/pokemon-blip-captions', split='train')
print(dataset, dataset.column_names)
for i in range(3):
    row = dataset[i]; print(row['text']); display(row['image'].convert('RGB').resize((256,256)))


## 2. Acceso al modelo base y configuración

`stable-diffusion-v1-5/stable-diffusion-v1-5` requiere aceptar la licencia del modelo y autenticarse. Este cuaderno usa el script oficial `train_text_to_image_lora.py`; la configuración limita resolución y pasos para caber en una GPU Colab. Si la memoria alcanza, se puede elevar resolución/entrenamiento después de registrar la primera ejecución.

In [ ]:
MODEL='stable-diffusion-v1-5/stable-diffusion-v1-5'
DATASET='lambdalabs/pokemon-blip-captions'
OUTPUT='/content/carlos-lora-pokemon'
SCRIPT='/content/diffusers/examples/text_to_image/train_text_to_image_lora.py'
assert os.path.exists(SCRIPT), 'No se encontró el script oficial de entrenamiento'


In [ ]:
!accelerate launch $SCRIPT \
  --pretrained_model_name_or_path=$MODEL \
  --dataset_name=$DATASET \
  --image_column=image \
  --caption_column=text \
  --resolution=256 \
  --center_crop --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --learning_rate=1e-4 \
  --lr_scheduler=constant \
  --lr_warmup_steps=0 \
  --max_train_steps=300 \
  --checkpointing_steps=100 \
  --validation_prompt='a hand-painted fantasy creature beside a mountain lake, storybook illustration' \
  --num_validation_images=2 \
  --validation_steps=100 \
  --mixed_precision=fp16 \
  --output_dir=$OUTPUT \
  --seed=1337


## 3. Comparación antes y después

Se conservan fijo el prompt, la semilla, el modelo base y los pasos de inferencia. Las diferencias visuales se interpretan como efecto del adaptador en este conjunto, no como una medición objetiva de superioridad. No se presenta una imagen de salida hasta que se complete el bloque de entrenamiento.

In [ ]:
from diffusers import StableDiffusionPipeline
from peft import PeftModel
from pathlib import Path

device='cuda'; dtype=torch.float16
prompt='a hand-painted fantasy creature beside a mountain lake, storybook illustration'
base=StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=dtype, safety_checker=None).to(device)
base.set_progress_bar_config(disable=True)
base_image=base(prompt, num_inference_steps=25, guidance_scale=7.0, generator=torch.Generator(device=device).manual_seed(42)).images[0]
lora=StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=dtype, safety_checker=None).to(device)
lora.load_lora_weights(OUTPUT)
lora.set_progress_bar_config(disable=True)
lora_image=lora(prompt, num_inference_steps=25, guidance_scale=7.0, generator=torch.Generator(device=device).manual_seed(42)).images[0]
os.makedirs('/content/resultados_lora_carlos', exist_ok=True)
base_image.save('/content/resultados_lora_carlos/base.png'); lora_image.save('/content/resultados_lora_carlos/lora.png')
fig, ax=plt.subplots(1,2,figsize=(10,5)); ax[0].imshow(base_image); ax[0].set_title('Modelo base'); ax[1].imshow(lora_image); ax[1].set_title('Base + LoRA');
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig('/content/resultados_lora_carlos/comparacion.png',dpi=160); plt.show()


## 4. Registro y limitaciones

Anote en la tabla siguiente la GPU que Colab asignó, tiempo total, pérdida final reportada por el script, errores y observación visual. El entrenamiento depende de disponibilidad de GPU, acceso al checkpoint y consumo de cuota. Las imágenes resultantes deben etiquetarse como generadas con IA. No publique token, pesos derivados si sus licencias no lo permiten, ni imágenes cuya licencia no autorice la redistribución.

In [ ]:
import json, datetime
registro={'fecha_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'gpu':torch.cuda.get_device_name(0),'base_model':MODEL,'dataset':DATASET,'resolution':256,'steps':300,'seed':1337,'prompt':prompt,'observacion':'Completar después de revisar base.png, lora.png y la salida del entrenamiento.'}
with open('/content/resultados_lora_carlos/registro.json','w',encoding='utf-8') as f: json.dump(registro,f,ensure_ascii=False,indent=2)
print(json.dumps(registro,ensure_ascii=False,indent=2))


## Referencias

Bouchard, L.-F. (2022, 28 de agosto). Latent diffusion models: The architecture behind Stable Diffusion. *Towards AI*. https://pub.towardsai.net/latent-diffusion-models-the-architecture-behind-stable-diffusion-434ba7d91108

Hugging Face. (s. f.). *Text-to-image fine-tuning*. Diffusers documentation. https://huggingface.co/docs/diffusers/en/training/text2image

Pinkney, J. N. M. (2022). *Pokémon BLIP captions* [Conjunto de datos]. Hugging Face. https://huggingface.co/datasets/lambdalabs/pokemon-blip-captions

Rombach, R., Blattmann, A., Lorenz, D., Esser, P., & Ommer, B. (2022). High-resolution image synthesis with latent diffusion models. *Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition*, 10684–10695. https://doi.org/10.1109/CVPR52688.2022.01042